# Customer Churn: Complete Analysis in Google Colab
### Segmentation + chi-square tests + prediction model

**What this notebook does.** It runs a complete churn analysis for the `Customer-Churn-Records.csv` dataset (10,000 customers):

| Part | Question it answers | What you get |
|---|---|---|
| **A. Setup** | Is the data clean? | Upload, validation, cleaning, baseline churn rate |
| **B. Segments and chi-square** | *Who* churns? | A segment table and chi-square test for every feature, one by one |
| **B2. Deeper analysis** | *How* do churners differ? | Churned vs retained profile, country x age, financial stability, high-value customers, revenue risk, the five KPIs |
| **C. Prediction model** | *Who will leave next?* | A trained, tested and evaluated churn model |
| **D. Action plan** | *What do we do for each customer?* | A ranked list with a suggested action per at-risk customer |

**How to run in Colab:** click **Runtime → Run all**, or press **Shift + Enter** on each cell. When Step 1 asks, upload `Customer-Churn-Records.csv`.

---
# Part A. Setup
## Step 1. Libraries
Colab already has all of these installed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             confusion_matrix, classification_report, precision_score, recall_score, f1_score)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (8, 3.8), "axes.spines.top": False, "axes.spines.right": False})
BLUE, GREY, RED = "#2a78d6", "#b9b8b0", "#d03b3b"
SEED = 42

## Step 2. Load the data
In Colab, a file picker opens: choose `Customer-Churn-Records.csv`.

In [ ]:
try:
    from google.colab import files          # only exists inside Colab
    uploaded = files.upload()               # choose Customer-Churn-Records.csv
    FILE = next(iter(uploaded))
except ImportError:
    FILE = "Customer-Churn-Records.csv"     # running outside Colab: file in the same folder

df = pd.read_csv(FILE)
# Normalize target column if named 'Exited'
if 'Exited' in df.columns and 'Churn' not in df.columns:
    df['Churn'] = df['Exited']

print(df.shape)
df.head()

## Step 3. Validate, clean and set the churn baseline
**Validation:** missing values, duplicate IDs, binary fields (`HasCrCard`, `IsActiveMember`, `Churn`) holding only 0/1, engagement and product fields inside plausible ranges, and a check that the churn label is correct.

**Cleaning:** the surname is removed because it identifies people and tells us nothing about churn.

**How to read it:** every `Out of range` count should be 0 and every 0/1 check should say `True`. A warning about `Complain` means it almost equals the churn label, so it is excluded from the model later (Step 14).

In [ ]:
id_col = "CustomerId" if "CustomerId" in df.columns else ("CustomerID" if "CustomerID" in df.columns else None)
print("Rows, columns:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
if id_col:
    print("Duplicate customer IDs:", int(df[id_col].duplicated().sum()))

print("\n1) Binary fields hold only 0/1")
for c in [c for c in ["HasCrCard", "IsActiveMember", "Exited", "Churn", "Complain"] if c in df.columns]:
    print(f"   {c:15s} {set(df[c].dropna().unique()) <= {0, 1}}")

print("\n2) Engagement and product fields are in a plausible range")
limits = {"Age": (18, 100), "CreditScore": (300, 900), "Tenure": (0, 50), "Balance": (0, None), "NumOfProducts": (1, 4),
          "EstimatedSalary": (0, None), "Satisfaction Score": (1, 5), "Point Earned": (0, None)}
rows = []
for c, (lo, hi) in limits.items():
    if c in df.columns:
        bad = int((df[c] < lo).sum() + ((df[c] > hi).sum() if hi is not None else 0))
        rows.append({"Field": c, "Min": df[c].min(), "Max": df[c].max(),
                     "Expected": f"{lo} to {hi if hi is not None else 'no limit'}", "Out of range": bad})
display(pd.DataFrame(rows))

print("3) Churn label")
if "Exited" in df.columns:
    print("   Churn equals Exited in every row:", bool((df.Churn == df.Exited).all()))
print("   Churn only 0/1:", set(df.Churn.unique()) <= {0, 1})
if "Complain" in df.columns and (df.Complain == df.Churn).mean() > 0.99:
    print("   WARNING: Complain matches Churn in over 99% of rows. It is almost the same field as the target.")

df = df.drop(columns=["Surname"], errors="ignore")       # cleaning: remove the non-analytical field
print("\nSurname removed:", "Surname" not in df.columns)

OVERALL = df.Churn.mean() * 100
print(f"\nOverall churn rate: {OVERALL:.1f}%  ({int(df.Churn.sum()):,} of {len(df):,} customers)")
BASE_RATE = df.Churn.mean()

---
# Part B. Who churns: segment tables and chi-square tests

## Step 4. Define the reusable functions
Run once. These functions build a segment table, run the chi-square test and draw a chart for any feature. Each step below just calls `analyze(...)`.

In [ ]:
RESULTS = []

def make_segments(feature, bins=None, labels=None, quantiles=None):
    s = df[feature]
    if bins is not None:
        return pd.cut(s, bins=bins, labels=labels)
    if quantiles is not None:
        return pd.qcut(s, q=quantiles, duplicates="drop")
    return s.astype("category")

def segment_table(seg):
    g = df.groupby(seg, observed=True).agg(
        Customers=("Churn", "size"),
        Churned=("Churn", "sum"),
        ChurnRate=("Churn", "mean"),
        AvgBalance=("Balance", "mean"),
    )
    g["BalanceLost"] = df[df.Churn == 1].groupby(seg[df.Churn == 1], observed=True).Balance.sum()
    g["BalanceLost"] = g.BalanceLost.fillna(0)
    g["ChurnRate"] = g.ChurnRate * 100
    g["ShareOfCustomers"] = g.Customers / len(df) * 100
    g["ShareOfChurners"] = g.Churned / df.Churn.sum() * 100
    g["RiskIndex"] = g.ChurnRate / OVERALL * 100
    g.index.name = seg.name
    return g[["Customers", "Churned", "ChurnRate", "ShareOfCustomers", "ShareOfChurners",
              "RiskIndex", "AvgBalance", "BalanceLost"]].round(1)

def chi2_test(seg):
    ct = pd.crosstab(seg, df.Churn)
    chi2, p, dof, expected = chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    low_share = (expected < 5).mean() * 100
    return {"chi2": round(chi2, 2), "dof": int(dof), "p_value": p, "cramers_v": round(v, 3),
            "min_expected": round(expected.min(), 1), "cells_below_5_pct": round(low_share, 1)}

def strength(v):
    return "negligible" if v < 0.05 else "weak" if v < 0.10 else "moderate" if v < 0.20 else "strong"

def analyze(feature, bins=None, labels=None, quantiles=None, show_chart=True):
    seg = make_segments(feature, bins, labels, quantiles)
    seg.name = feature
    table = segment_table(seg)
    test = chi2_test(seg)
    significant = test["p_value"] < 0.05
    reliable = test["cells_below_5_pct"] <= 20

    print(f"=== {feature} ===")
    display(table)
    print(f"Chi-square = {test['chi2']}, dof = {test['dof']}, p-value = {test['p_value']:.4g}, "
          f"Cramer's V = {test['cramers_v']} ({strength(test['cramers_v'])})")
    verdict = ("churn DIFFERS across segments (significant)" if significant
               else "no significant difference in churn across segments")
    print("Verdict:", verdict)
    if not reliable:
        print(f"Caution: {test['cells_below_5_pct']}% of cells have expected counts below 5.")

    if show_chart:
        colors = [BLUE if r > OVERALL else GREY for r in table.ChurnRate]
        ax = table.ChurnRate.plot(kind="bar", color=colors, rot=20 if len(table) > 5 else 0)
        ax.axhline(OVERALL, ls="--", color="black", lw=1)
        ax.set_ylabel("Churn rate (%)"); ax.set_xlabel(""); ax.set_title(f"Churn rate by {feature} (dashed = overall {OVERALL:.1f}%)")
        for i, v in enumerate(table.ChurnRate):
            ax.text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=9)
        plt.tight_layout(); plt.show()

    RESULTS.append({"Feature": feature, "Segments": len(table), "ChurnRateMin": table.ChurnRate.min(),
                    "ChurnRateMax": table.ChurnRate.max(), **test,
                    "Significant": "Yes" if significant else "No", "Strength": strength(test["cramers_v"]),
                    "Reliable": "Yes" if reliable else "No"})
    return table

## Step 5. Geography

In [ ]:
analyze("Geography")

## Step 6. Gender

In [ ]:
analyze("Gender")

## Step 7. Age

In [ ]:
analyze("Age", bins=[0, 29, 45, 60, 200], labels=["<30", "30-45", "46-60", "60+"])

## Step 8. Tenure

In [ ]:
analyze("Tenure", bins=[-1, 2, 6, 100], labels=["New (0-2y)", "Mid-term (3-6y)", "Long-term (7y+)"])

## Step 9. CreditScore

In [ ]:
analyze("CreditScore", bins=[0, 579, 739, 900], labels=["Low (<580)", "Medium (580-739)", "High (740+)"])

## Step 10. Balance

In [ ]:
df["BalanceGroup"] = np.select(
    [df.Balance == 0, (df.Balance > 0) & (df.Balance <= 50000), (df.Balance > 50000) & (df.Balance <= 100000), df.Balance > 100000],
    ["Zero", "Low (1-50k)", "Medium (50-100k)", "High (100k+)"],
    default="Unknown"          # added: np.select needs a text default
)
analyze("BalanceGroup")

## Step 11. Complaints (if present)

In [ ]:
if "Complain" in df.columns:
    analyze("Complain")

## Step 12. Summary of all Chi-Square Tests

In [ ]:
summary_df = pd.DataFrame(RESULTS).sort_values(by="cramers_v", ascending=False)
display(summary_df)

---
# Part B2. Deeper analysis: profiles, interactions, high-value customers and KPIs
Part B tested each feature on its own. Part B2 completes the analysis plan: how churners differ from retained customers, how country and age interact, financial stability, high-value customers and revenue at risk, and the five KPIs.

| Step | What it does |
|---|---|
| 12A | Create the grouped fields used below |
| 12B | Churned vs retained customers: side-by-side profile |
| 12C | Geography and age interaction |
| 12D | Financial stability vs churn |
| 12E | High-value customers: high-balance churners, salary vs balance, revenue risk |
| 12F | The five KPIs |

## Step 12A. Create the grouped fields
Adds the age, credit, tenure and salary groups, and an Active / Inactive label, to the data, using the same cut points as Part B. `BalanceGroup` already exists from Step 10; here it is put in a fixed order from Zero to High.

In [ ]:
df["AgeGroup"] = pd.cut(df.Age, bins=[0, 29, 45, 60, 200], labels=["<30", "30-45", "46-60", "60+"])
df["CreditBand"] = pd.cut(df.CreditScore, bins=[0, 579, 739, 900], labels=["Low (<580)", "Medium (580-739)", "High (740+)"])
df["TenureGroup"] = pd.cut(df.Tenure, bins=[-1, 2, 6, 100], labels=["New (0-2y)", "Mid-term (3-6y)", "Long-term (7y+)"])
df["SalaryQuartile"] = pd.qcut(df.EstimatedSalary, 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
df["Activity"] = np.where(df.IsActiveMember == 1, "Active", "Inactive")
BAL_ORDER = ["Zero", "Low (1-50k)", "Medium (50-100k)", "High (100k+)"]
df["BalanceGroup"] = pd.Categorical(df.BalanceGroup, categories=BAL_ORDER, ordered=True)

for c in ["AgeGroup", "CreditBand", "TenureGroup", "BalanceGroup", "Activity"]:
    print(c, df[c].value_counts().to_dict())

## Step 12B. Churned vs retained customers
Puts the two groups side by side.

- **Numeric fields:** the average for each group and the % difference (churned vs retained).
- **Category fields:** the share of each group in every category, and the gap in percentage points.

**How to read it:** a large positive difference means churners have *more* of this (for example a higher average age). A large gap in shares means that category is over-represented among churners.

In [ ]:
num = [c for c in ["Age", "CreditScore", "Tenure", "Balance", "EstimatedSalary", "NumOfProducts",
                   "Satisfaction Score", "Point Earned"] if c in df.columns]
prof = df.groupby("Churn")[num].mean().T.rename(columns={0: "Retained", 1: "Churned"})
prof["Difference %"] = (prof.Churned / prof.Retained - 1) * 100
display(prof.round(2))

ax = prof["Difference %"].sort_values().plot(kind="barh", figsize=(7, 3.6),
        color=[BLUE if v > 0 else GREY for v in prof["Difference %"].sort_values()])
ax.axvline(0, color="black", lw=0.8); ax.set_xlabel("Churned vs retained, % difference in the average")
ax.set_title("How churners differ from retained customers"); plt.tight_layout(); plt.show()

cats = [c for c in ["Geography", "Gender", "Activity", "HasCrCard", "NumOfProducts", "Complain"] if c in df.columns]
parts = []
for c in cats:
    t = pd.crosstab(df[c], df.Churn, normalize="columns") * 100
    t.columns = ["Retained %", "Churned %"]
    t["Gap (points)"] = t["Churned %"] - t["Retained %"]
    t.index = [f"{c} = {i}" for i in t.index]
    parts.append(t)
display(pd.concat(parts).round(1))

## Step 12C. Geography and age interaction
Does the effect of age differ from country to country? The table and heatmap show the churn rate for every country and age group together. Cells with fewer than 30 customers are blanked because their rates are unreliable.

**How to read it:** if each country shows the same pattern across age groups, there is no real interaction. A cell much darker than its row and column neighbours marks a specific pocket of risk. The last table shows, per country, the gap between its highest and lowest age group.

In [ ]:
def heatmap(pt, cnt, title, min_n=30):
    data = pt.where(cnt >= min_n)
    fig, ax = plt.subplots(figsize=(7.5, 0.9 * len(pt) + 1.8))
    im = ax.imshow(data.values.astype(float), cmap="Reds", vmin=0)
    ax.set_xticks(range(data.shape[1]), [str(c) for c in data.columns])
    ax.set_yticks(range(data.shape[0]), [str(i) for i in data.index])
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data.values[i, j]
            ax.text(j, i, "n<30" if pd.isna(v) else f"{v:.1f}%\n(n={int(cnt.values[i, j]):,})", ha="center", va="center", fontsize=9)
    ax.set_title(title); plt.colorbar(im, ax=ax, label="Churn rate %")
    plt.tight_layout(); plt.show()

def churn_grid(row, col):
    pt = df.pivot_table(index=row, columns=col, values="Churn", aggfunc="mean", observed=True) * 100
    cnt = df.pivot_table(index=row, columns=col, values="Churn", aggfunc="size", observed=True).fillna(0)
    return pt, cnt

pt, cnt = churn_grid("Geography", "AgeGroup")
display(pt.round(1))
heatmap(pt, cnt, "Churn rate by country and age group")

spread = pt.where(cnt >= 30)
gap = pd.DataFrame({"Highest": spread.max(axis=1), "Lowest": spread.min(axis=1)})
gap["Gap (points)"] = gap.Highest - gap.Lowest
gap["Highest age group"] = spread.idxmax(axis=1)
display(gap.round(1))

## Step 12D. Financial stability vs churn
Financial stability is read from three signals: **credit score**, **balance** and **products held**. The heatmap crosses credit band with balance group, and the second table crosses credit band with number of products.

**How to read it:** look for the darkest cells. If churn is high for low credit *and* zero balance but not for either alone, the combination is what matters. If the colours barely change, financial standing does not explain churn much.

In [ ]:
pt, cnt = churn_grid("CreditBand", "BalanceGroup")
display(pt.round(1))
heatmap(pt, cnt, "Churn rate by credit band and balance group")

pt2, cnt2 = churn_grid("CreditBand", "NumOfProducts")
display(pt2.round(1))

## Step 12E. High-value customers: high-balance churners, salary vs balance, revenue risk
Defines **high-value** as a balance above `HIGH_VALUE_BALANCE` (change it if your bank uses another line).

1. **High-balance churners:** how many high-balance customers left, what share of all churners and of all lost money they are, and the 10 largest.
2. **Salary vs balance:** churn rate by salary quartile next to churn rate by balance group. If balance moves churn but salary does not, balance is the better lens.
3. **Revenue risk:** the money that left, and where it came from.

> The data has **no revenue or profit field**, so money at risk means the **account balance** that left. `ASSUMED_MARGIN` is a **made-up illustration** (2% of balance as yearly income). Replace it with your bank's real figure or ignore that column.

In [ ]:
HIGH_VALUE_BALANCE = 100_000
ASSUMED_MARGIN = 0.02        # ILLUSTRATION ONLY: yearly income as a share of balance. Replace with your own figure.

hv = df[df.Balance > HIGH_VALUE_BALANCE]
ch = df[df.Churn == 1]
hv_ch = hv[hv.Churn == 1]
summary = pd.Series({
    "High-balance customers": len(hv),
    "High-balance share of all customers %": len(hv) / len(df) * 100,
    "High-balance churners": len(hv_ch),
    "High-balance churn rate %": hv.Churn.mean() * 100,
    "Overall churn rate %": OVERALL,
    "Share of all churners %": len(hv_ch) / len(ch) * 100,
    "Balance lost from high-balance churners": hv_ch.Balance.sum(),
    "Share of all lost balance %": hv_ch.Balance.sum() / ch.Balance.sum() * 100,
})
display(summary.round(1).to_frame("Value"))

print("The 10 largest balances among customers who left:")
show = [c for c in [id_col, "Geography", "Age", "Balance", "EstimatedSalary", "NumOfProducts", "Activity"] if c and c in df.columns]
display(ch.sort_values("Balance", ascending=False)[show].head(10).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col, title in [(axes[0], "SalaryQuartile", "By salary quartile"), (axes[1], "BalanceGroup", "By balance group")]:
    r = df.groupby(col, observed=True).Churn.mean() * 100
    ax.bar(r.index.astype(str), r.values, color=BLUE)
    ax.axhline(OVERALL, ls="--", color="black", lw=1)
    ax.set_title(title); ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(r.values): ax.text(i, v + 0.4, f"{v:.1f}%", ha="center")
axes[0].set_ylabel("Churn rate %")
plt.suptitle("Salary vs balance: which one moves churn? (dashed = overall)"); plt.tight_layout(); plt.show()
print("Correlation between balance and salary:", round(df.Balance.corr(df.EstimatedSalary), 3))

total_bal, lost_bal = df.Balance.sum(), ch.Balance.sum()
print(f"\nBalance that left: {lost_bal/1e6:.1f}M of {total_bal/1e6:.1f}M ({lost_bal/total_bal*100:.1f}% of all balance)")
print(f"Illustrative yearly income at risk at a {ASSUMED_MARGIN*100:.0f}% margin: {lost_bal*ASSUMED_MARGIN/1e6:.2f}M (assumption, not data)")

rows = []
for col in ["Geography", "Activity", "BalanceGroup"]:
    g = df.groupby(col, observed=True).agg(Customers=("Churn", "size"), Churned=("Churn", "sum"), ChurnRate=("Churn", "mean"))
    g["BalanceLost"] = ch.groupby(col, observed=True).Balance.sum()
    g["ShareOfBalanceLost %"] = g.BalanceLost / lost_bal * 100
    g["IllustrativeRevenueAtRisk"] = g.BalanceLost * ASSUMED_MARGIN
    g["ChurnRate"] = g.ChurnRate * 100
    g.insert(0, "Dimension", col)
    rows.append(g)
display(pd.concat(rows).round(1))

## Step 12F. The five KPIs
| KPI | Definition used here |
|---|---|
| **Overall Churn Rate** | customers who exited ÷ all customers |
| **Segment Churn Rate** | churn % for each segment, with the highest and lowest segment per dimension |
| **High-Value Churn Ratio** | churn rate of high-value customers ÷ overall churn rate (above 1 means premium customers leave *more* than average). Premium = balance above `HIGH_VALUE_BALANCE`, and also platinum / diamond cardholders if the file has `Card Type` |
| **Geographic Risk Index** | a country's churn rate ÷ overall churn rate × 100 (100 = average, above 100 = riskier) |
| **Engagement Drop Indicator** | churn rate of inactive members ÷ churn rate of active members, plus the extra churners that inactivity accounts for |

**How to read it:** the summary table at the end gives all five on one screen. These KPIs describe association, not cause.

In [ ]:
# KPI 2: segment churn rates across every dimension
dims = [c for c in ["Geography", "Gender", "AgeGroup", "CreditBand", "TenureGroup", "BalanceGroup", "Activity"] if c in df.columns]
rows = []
for c in dims:
    g = df.groupby(c, observed=True).Churn.agg(["mean", "size"])
    for seg, r in g.iterrows():
        rows.append({"Dimension": c, "Segment": str(seg), "Customers": int(r["size"]), "ChurnRate %": r["mean"] * 100})
seg_kpi = pd.DataFrame(rows)
hi = seg_kpi.loc[seg_kpi.groupby("Dimension")["ChurnRate %"].idxmax()].set_index("Dimension")
lo = seg_kpi.loc[seg_kpi.groupby("Dimension")["ChurnRate %"].idxmin()].set_index("Dimension")
seg_summary = pd.DataFrame({"Highest segment": hi.Segment, "Highest %": hi["ChurnRate %"],
                            "Lowest segment": lo.Segment, "Lowest %": lo["ChurnRate %"]})
seg_summary["Spread (points)"] = seg_summary["Highest %"] - seg_summary["Lowest %"]
print("KPI 2. Segment churn rate: highest and lowest segment in each dimension")
display(seg_summary.sort_values("Spread (points)", ascending=False).round(1))

# KPI 3: high-value churn ratio
premium = {"Balance above " + f"{HIGH_VALUE_BALANCE:,}": df.Balance > HIGH_VALUE_BALANCE}
if "Card Type" in df.columns:
    card_mask = df["Card Type"].astype(str).str.upper().isin(["PLATINUM", "DIAMOND"])
    if card_mask.sum() > 0:
        premium["Platinum / Diamond card"] = card_mask
hv_rows = []
for name, m in premium.items():
    rate = df.loc[m, "Churn"].mean() * 100
    hv_rows.append({"Premium definition": name, "Customers": int(m.sum()), "Churn rate %": rate,
                    "High-Value Churn Ratio": rate / OVERALL})
hv_kpi = pd.DataFrame(hv_rows)
print("\nKPI 3. High-Value Churn Ratio")
display(hv_kpi.round(2))

# KPI 4: geographic risk index
geo = df.groupby("Geography").agg(Customers=("Churn", "size"), Churned=("Churn", "sum"), ChurnRate=("Churn", "mean"))
geo["ChurnRate"] = geo.ChurnRate * 100
geo["Geographic Risk Index"] = geo.ChurnRate / OVERALL * 100
geo["Share of churners %"] = geo.Churned / geo.Churned.sum() * 100
print("\nKPI 4. Geographic Risk Index (100 = average)")
display(geo.round(1))
ax = geo["Geographic Risk Index"].plot(kind="bar", color=BLUE, rot=0, figsize=(5.5, 3.2))
ax.axhline(100, ls="--", color="black", lw=1); ax.set_ylabel("Index"); ax.set_xlabel("")
ax.set_title("Geographic Risk Index (dashed = average)")
for i, v in enumerate(geo["Geographic Risk Index"]): ax.text(i, v + 1, f"{v:.0f}", ha="center")
plt.tight_layout(); plt.show()

# KPI 5: engagement drop indicator
act = df.groupby("Activity").Churn.agg(["mean", "size"])
inactive_rate, active_rate = act.loc["Inactive", "mean"] * 100, act.loc["Active", "mean"] * 100
extra = (inactive_rate - active_rate) / 100 * act.loc["Inactive", "size"]
print(f"\nKPI 5. Engagement Drop Indicator: inactive {inactive_rate:.1f}% vs active {active_rate:.1f}% "
      f"(ratio {inactive_rate/active_rate:.2f}x); about {extra:.0f} extra churners are linked to inactivity")

# All five on one screen
geo_text = ", ".join(f"{k} {v:.0f}" for k, v in geo["Geographic Risk Index"].items())
kpi_table = pd.DataFrame([
    ["Overall Churn Rate", f"{OVERALL:.1f}%", "customers who exited"],
    ["Segment Churn Rate", f"{seg_kpi['ChurnRate %'].min():.1f}% to {seg_kpi['ChurnRate %'].max():.1f}%", "lowest to highest segment"],
    ["High-Value Churn Ratio", " | ".join(f"{r['High-Value Churn Ratio']:.2f}" for _, r in hv_kpi.iterrows()), "premium churn rate / overall (1 = same as average)"],
    ["Geographic Risk Index", geo_text, "country churn rate / overall x 100"],
    ["Engagement Drop Indicator", f"{inactive_rate/active_rate:.2f}x ({inactive_rate:.1f}% vs {active_rate:.1f}%)", "inactive churn rate / active churn rate"],
], columns=["KPI", "Value", "Meaning"])
print("\nThe five KPIs")
display(kpi_table)

---
# Part C. Prediction models: train, compare and check errors
Part B showed which features are linked to churn. Part C trains **five different models**, compares them on the same hidden test customers, and uses **confusion matrices** to count the mistakes each one makes. Part D then finds the **high-risk customers** and saves them to a CSV.

| Step | What it does |
|---|---|
| 13 | Choose the features |
| 14 | Leakage check: remove any column that gives away the answer |
| 15 | Prepare the data and split train / test |
| 16 | Define five models |
| 17 | Train all five and score them on the test set |
| 18 | Compare the models in one table and chart |
| 19 | Confusion matrix for the best model |
| 20 | How the threshold changes false positives and false negatives |
| 21 | ROC curves, and the final model |
| 22 | SHAP: which features push risk up or down |

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score

## Step 13. Choose the features
Removes the columns the model must not use: row and customer identifiers, the surname, and the target itself (`Churn`, and its twin `Exited`).

In [ ]:
ID_AND_TARGET = [c for c in ["RowNumber", "CustomerId", "CustomerID", "Surname", "Churn", "Exited", "BalanceGroup", "AgeGroup", "CreditBand", "TenureGroup", "SalaryQuartile", "Activity"] if c in df.columns]
X = df.drop(columns=ID_AND_TARGET)
y = df["Churn"]
print("Removed:", ID_AND_TARGET)
print("Candidate features:", X.columns.tolist())

## Step 14. Leakage check
A column that almost *equals* the answer makes a model look perfect (an AUC near 1.0) without being useful. This is **leakage**: using information the bank would not have *before* a customer decides to leave.

**What the code does:** for every feature it measures how strongly it is tied to churn on its own: **Cramér's V** for categories and yes/no fields, and **single-feature AUC** for numbers. Anything above the limits (V above 0.90 or AUC above 0.95) is flagged and removed.

**How to read it:** the table lists the five strongest single features. If a column is flagged (for example a `Complain` field that is nearly identical to `Exited`), it is dropped automatically. A normal feature has a V below about 0.25.

In [ ]:
def leakage_check(X, y, v_limit=0.90, auc_limit=0.95):
    rows = []
    for c in X.columns:
        s = X[c]
        if s.dtype.kind in "if" and s.nunique() > 10:
            a = roc_auc_score(y, s)
            score, kind, limit = max(a, 1 - a), "single-feature AUC", auc_limit
        else:
            ct = pd.crosstab(s, y)
            if ct.shape[0] < 2:
                continue
            chi2 = chi2_contingency(ct)[0]
            score, kind, limit = np.sqrt(chi2 / (len(y) * (min(ct.shape) - 1))), "Cramers V", v_limit
        rows.append({"Feature": c, "Measure": kind, "Score": round(score, 3), "Limit": limit, "Leaky": score > limit})
    return pd.DataFrame(rows).sort_values("Score", ascending=False).reset_index(drop=True)

leak = leakage_check(X, y)
display(leak.head(5))
LEAKY = leak.loc[leak.Leaky, "Feature"].tolist()
if LEAKY:
    print("LEAKAGE FOUND, removing:", LEAKY)
    X = X.drop(columns=LEAKY)
else:
    print("No leaky features found.")
print("Features used by the models:", X.columns.tolist())

## Step 15. Prepare the data and split train / test
- **Numeric** columns are scaled and **text** columns are one-hot encoded inside a *pipeline*, so the same preparation is applied automatically whenever the model is used.
- **20% of customers are hidden** as a test set. `stratify=y` keeps the churn rate the same in both parts.

> Never choose or tune anything using the test set. It is only for the final honest score.

In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

def make_preprocessor(scale=True):
    return ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"\nTrain: {len(X_train):,} customers (churn {y_train.mean()*100:.1f}%) | Test: {len(X_test):,} customers (churn {y_test.mean()*100:.1f}%)")

## Step 16. Define five models
Five different kinds of model are compared, so the result does not depend on one method:

| Model | Idea | Why include it |
|---|---|---|
| Logistic Regression | a weighted score added up | simple and easy to explain |
| Decision Tree | a chain of yes/no questions | very readable rules |
| Random Forest | many trees voting | handles complex patterns |
| Gradient Boosting | trees built one after another to fix mistakes | often the most accurate on tables |
| K-Nearest Neighbors | looks at the 25 most similar customers | a very different approach |

`class_weight="balanced"` tells the first three models to pay extra attention to churners, who are the minority.

In [ ]:
models = {
    "Logistic Regression": Pipeline([("prep", make_preprocessor(True)),
                                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "Decision Tree": Pipeline([("prep", make_preprocessor(False)),
                               ("clf", DecisionTreeClassifier(max_depth=5, min_samples_leaf=30,
                                                              class_weight="balanced", random_state=SEED))]),
    "Random Forest": Pipeline([("prep", make_preprocessor(False)),
                               ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                              class_weight="balanced", n_jobs=-1, random_state=SEED))]),
    "Gradient Boosting": Pipeline([("prep", make_preprocessor(False)),
                                   ("clf", GradientBoostingClassifier(random_state=SEED))]),
    "K-Nearest Neighbors": Pipeline([("prep", make_preprocessor(True)),
                                     ("clf", KNeighborsClassifier(n_neighbors=25))]),
}
print("Models:", list(models))

## Step 17. Train all five and score them on the test set
For each model the code:
1. runs **5-fold cross-validation** on the training data, which gives a stable AUC;
2. **trains** on the full training set;
3. predicts churn probabilities for the hidden test customers and turns them into yes/no with a **threshold** (default 0.5);
4. counts the four outcomes of the **confusion matrix**.

| Outcome | Meaning |
|---|---|
| **TN** true negative | predicted to stay, and stayed |
| **FP** false positive | predicted to leave, but **stayed** (a wasted retention offer) |
| **FN** false negative | predicted to stay, but **left** (a churner we missed) |
| **TP** true positive | predicted to leave, and left (a churner caught) |

Change `THRESHOLD` to see how the mistakes shift.

In [ ]:
THRESHOLD = 0.5
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

test_prob, rows = {}, []
for name, pipe in models.items():
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    pred = (p >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    test_prob[name] = p
    rows.append({
        "Model": name,
        "CV AUC": cv_auc,
        "Train AUC": roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1]),
        "Test AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "FalsePositiveRate": fp / (fp + tn),
        "Specificity (TN rate)": tn / (tn + fp),
    })
    print(f"trained {name}")

comparison = pd.DataFrame(rows).set_index("Model")

## Step 18. Compare the models
All five models side by side on the same hidden test customers.

**How to read the columns**
- **Test AUC:** how well the model ranks churners above stayers (0.5 = guessing, 1.0 = perfect). **The main score, because the classes are unequal.**
- **Train AUC vs Test AUC:** a big gap means **overfitting** (memorising the training data).
- **Precision:** of customers flagged as leaving, the share who really leave.
- **Recall:** of customers who really leave, the share we catch.
- **Accuracy** is shown for completeness but can mislead: predicting "nobody leaves" is already about 80% accurate when churn is about 20%.

The best model is chosen by **Test AUC**.

In [ ]:
metric_cols = ["CV AUC", "Train AUC", "Test AUC", "PR-AUC", "Accuracy", "Precision", "Recall", "F1"]
display(comparison[metric_cols].sort_values("Test AUC", ascending=False).round(3))

best_name = comparison["Test AUC"].idxmax()
print(f"Best model by Test AUC: {best_name} ({comparison.loc[best_name, 'Test AUC']:.3f})")

ax = comparison[["Test AUC", "Precision", "Recall", "F1"]].plot(kind="bar", figsize=(10, 4), rot=15,
        color=["#2a78d6", "#b9b8b0", "#eb6834", "#1baf7a"])
ax.set_title("Model comparison on the hidden test set"); ax.set_xlabel(""); ax.set_ylim(0, 1)
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.tight_layout(); plt.show()

## Step 19. Confusion matrix for the best model
Shows the four counts for the **best model only** (chosen in Step 18), as a heatmap and a short table, with the two rates that matter most for false alarms:

- **False Positive Rate (FPR)** = FP ÷ (FP + TN): of customers who **stayed**, the share wrongly flagged as leaving.
- **Specificity (TN rate)** = TN ÷ (TN + FP): of customers who **stayed**, the share correctly left alone. It equals 1 − FPR.

**How to read it:** the top-left cell is TN, top-right is **FP**, bottom-left is **FN**, bottom-right is TP. A good model has large TN and TP and small FP and FN. Many FP means retention offers wasted on customers who would have stayed. Many FN means churners we missed.

In [ ]:
r = comparison.loc[best_name]
cm = np.array([[r.TN, r.FP], [r.FN, r.TP]])
labels = [["TN", "FP"], ["FN", "TP"]]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]:,}", ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["Predicted stay", "Predicted leave"]); ax.set_yticks([0, 1], ["Actually stayed", "Actually left"])
ax.set_title(f"{best_name}: confusion matrix (threshold {THRESHOLD})")
plt.tight_layout(); plt.show()

print(f"True negatives  (stayed, left alone):          {int(r.TN):,}")
print(f"False positives (stayed, wrongly flagged):     {int(r.FP):,}  -> false positive rate {r.FalsePositiveRate*100:.1f}%")
print(f"False negatives (left, missed):                {int(r.FN):,}")
print(f"True positives  (left, correctly flagged):     {int(r.TP):,}  -> recall {r.Recall*100:.1f}%")
print(f"Specificity (share of stayers left alone):     {r['Specificity (TN rate)']*100:.1f}%")

## Step 20. How the threshold changes the mistakes (best model)
The threshold decides who is flagged. A **lower** threshold flags more customers: you catch more churners (fewer FN) but also raise more false alarms (more FP, fewer TN). A **higher** threshold does the opposite.

**How to read it:** pick the row that matches your budget. If a retention offer is cheap, accept more FP to catch more churners. If offers are expensive, choose a row with fewer FP. The row with the highest **F1** balances precision and recall.

In [ ]:
p_best = test_prob[best_name]
rows = []
for t in np.arange(0.10, 0.91, 0.05):
    pred = (p_best >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    rows.append({"Threshold": round(t, 2), "Flagged": int(pred.sum()), "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                 "Precision": round(precision_score(y_test, pred, zero_division=0), 3),
                 "Recall": round(recall_score(y_test, pred), 3), "F1": round(f1_score(y_test, pred), 3)})
thr_table = pd.DataFrame(rows)
display(thr_table)
BEST_F1_THRESHOLD = float(thr_table.loc[thr_table.F1.idxmax(), "Threshold"])
print(f"Threshold with the best F1 for {best_name}: {BEST_F1_THRESHOLD}")

## Step 21. ROC curves and the final model
The **ROC curve** shows how well each model separates churners from stayers: the further a curve rises above the diagonal, the better. The report below gives the best model's precision, recall and F1 for each class.

In [ ]:
plt.figure(figsize=(6.5, 5))
for name, p in test_prob.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=2.5 if name == best_name else 1.2, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "--", color="black", lw=1)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves (hidden test set)")
plt.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

final_model = models[best_name]
pred_best = (test_prob[best_name] >= THRESHOLD).astype(int)
print(f"Final model: {best_name} at threshold {THRESHOLD}")
print(classification_report(y_test, pred_best, target_names=["Stayed", "Left"], digits=3))

## Step 22. SHAP: why does the model score customers as risky?
**SHAP** explains a model by giving every feature a score for every customer: how much it **pushed that customer's churn risk up (positive) or down (negative)**. Averaged over many customers it shows which features matter most and in which direction.

**How to read the two charts**
- **Bar chart:** the features ranked by average impact. The longest bar matters most.
- **Dot chart (beeswarm):** one dot per customer. Dots to the **right** raised risk, dots to the left lowered it. Red dots are a **high** value of the feature, blue a **low** value. For example, red dots on the right for `Age` means older customers have higher risk.

SHAP is only needed to explain the model; it does not change any result above. It is computed on a sample of test customers to keep it fast. Like everything here, it shows **association, not cause**.

In [ ]:
import sys, subprocess
try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

prep, clf = final_model.named_steps["prep"], final_model.named_steps["clf"]

def to_dense(m):
    return m.toarray() if hasattr(m, "toarray") else np.asarray(m)

feat_names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
Xt_train = to_dense(prep.transform(X_train))
sample = X_test.sample(min(500, len(X_test)), random_state=SEED)
Xt = to_dense(prep.transform(sample))

if best_name == "Logistic Regression":
    explainer = shap.LinearExplainer(clf, Xt_train)
    sv = explainer.shap_values(Xt)
elif best_name == "K-Nearest Neighbors":
    background = shap.kmeans(Xt_train, 20)
    explainer = shap.KernelExplainer(lambda d: clf.predict_proba(d)[:, 1], background)
    sv = explainer.shap_values(Xt[:100], nsamples=100)
    Xt = Xt[:100]
else:
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(Xt)

if isinstance(sv, list):          # older shap: one array per class
    sv = sv[1]
sv = np.asarray(sv)
if sv.ndim == 3:                  # newer shap: (rows, features, classes)
    sv = sv[:, :, 1]

shap.summary_plot(sv, Xt, feature_names=feat_names, plot_type="bar", show=False)
plt.title(f"Average impact of each feature ({best_name})"); plt.tight_layout(); plt.show()

shap.summary_plot(sv, Xt, feature_names=feat_names, show=False)
plt.title("How feature values push risk up or down"); plt.tight_layout(); plt.show()

shap_importance = (pd.Series(np.abs(sv).mean(axis=0), index=feat_names)
                   .sort_values(ascending=False).rename("Mean |SHAP|").to_frame())
display(shap_importance.head(10).round(4))

---
# Part D. High-risk customers: find, analyse and export

## Step 23. Score every customer
Every customer gets a **churn probability** from the final model.

**Honest scoring:** `cross_val_predict` scores each customer using a model that **never saw that customer** during training. Scoring customers the model had memorised would inflate the results.

**Risk tiers** are set by rank, so they work whatever the model's probability scale:
- **High risk:** the top `HIGH_RISK_SHARE` of customers (default 20%).
- **Medium:** the next 30%.
- **Low:** the rest.

Change `HIGH_RISK_SHARE` to widen or narrow the high-risk group.

In [ ]:
HIGH_RISK_SHARE = 0.20
MEDIUM_RISK_SHARE = 0.30

oof = cross_val_predict(clone(models[best_name]), X, y, cv=cv, method="predict_proba")[:, 1]

scored = df.copy()
scored["ChurnProbability"] = oof.round(4)
rank_pct = scored.ChurnProbability.rank(pct=True, ascending=False, method="first")
scored["RiskTier"] = np.where(rank_pct <= HIGH_RISK_SHARE, "High",
                      np.where(rank_pct <= HIGH_RISK_SHARE + MEDIUM_RISK_SHARE, "Medium", "Low"))
scored["ExpectedBalanceAtRisk"] = (scored.ChurnProbability * scored.Balance).round(0)
print(scored.RiskTier.value_counts())

## Step 24. Does the risk tier work?
Checks the tiers against what **actually** happened.

**How to read it:** if the model works, the actual churn rate rises from Low to High. **Lift** compares each tier with the overall rate (a lift of 2 means twice the average churn). **% of all churners** shows how many real churners sit in each tier, so you can see how many are reached by contacting only the High tier.

In [ ]:
tier = scored.groupby("RiskTier").agg(
    Customers=("Churn", "size"), Churned=("Churn", "sum"), ActualChurnRate=("Churn", "mean"),
    AvgProbability=("ChurnProbability", "mean"), TotalBalance=("Balance", "sum"),
    ExpectedBalanceAtRisk=("ExpectedBalanceAtRisk", "sum"),
).loc[["High", "Medium", "Low"]]
tier["Lift"] = tier.ActualChurnRate / BASE_RATE
tier["% of all churners"] = tier.Churned / tier.Churned.sum() * 100
tier["ActualChurnRate"] = tier.ActualChurnRate * 100
display(tier.round(2))

ax = tier.ActualChurnRate.plot(kind="bar", color=["#d03b3b", "#eda100", "#b9b8b0"], rot=0, figsize=(6, 3.5))
ax.axhline(BASE_RATE * 100, ls="--", color="black", lw=1)
ax.set_ylabel("Actual churn rate (%)"); ax.set_xlabel(""); ax.set_title("Actual churn by predicted risk tier (dashed = overall)")
for i, v in enumerate(tier.ActualChurnRate): ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.tight_layout(); plt.show()

hi = scored[scored.RiskTier == "High"]
print(f"High-risk group: {len(hi):,} customers ({len(hi)/len(scored)*100:.0f}% of all)")
print(f"  catches {hi.Churn.sum()/scored.Churn.sum()*100:.1f}% of all churners; {hi.Churn.mean()*100:.1f}% of them actually left")
print(f"  total balance {hi.Balance.sum()/1e6:.1f}M; expected balance at risk {hi.ExpectedBalanceAtRisk.sum()/1e6:.1f}M")

## Step 25. Who are the high-risk customers?
Compares the **High-risk** group with everyone else.

1. **Numeric profile:** the average of each numeric field for High-risk and the rest, with the percentage difference.
2. **Over-represented segments:** categories that are **more common** among High-risk customers than in the whole base. `Ratio` above 1.2 means at least 20% over-represented.

**How to read it:** this tells you what high-risk customers have in common, so you can design the retention action.

In [ ]:
rest = scored[scored.RiskTier != "High"]
num_cols_profile = [c for c in numeric_features if c in scored.columns]
profile = pd.DataFrame({"High risk": hi[num_cols_profile].mean(), "Everyone else": rest[num_cols_profile].mean()})
profile["Difference %"] = (profile["High risk"] / profile["Everyone else"] - 1) * 100
display(profile.round(2))

cat_like = categorical_features + [c for c in numeric_features if scored[c].nunique() <= 8]
rows = []
for c in cat_like:
    share_hi = hi[c].value_counts(normalize=True)
    share_all = scored[c].value_counts(normalize=True)
    for seg, s in share_hi.items():
        rows.append({"Feature": c, "Segment": seg, "% of high-risk": s * 100, "% of all customers": share_all[seg] * 100,
                     "Ratio": s / share_all[seg], "HighRiskCustomers": int(round(s * len(hi)))})
over = pd.DataFrame(rows)
over = over[(over.Ratio >= 1.2) & (over.HighRiskCustomers >= 30)].sort_values("Ratio", ascending=False)
print("Segments over-represented among high-risk customers:")
display(over.round(2).reset_index(drop=True))

## Step 26. Why is each customer high risk? Add risk drivers
For each customer, lists the **risk drivers**: the segments they belong to where the actual churn rate is at least **1.25 times the average** (and the segment has at least 100 customers).

**Example:** `IsActiveMember=0; NumOfProducts=1` means the customer is inactive and holds one product, both segments with above-average churn.

> These drivers are **descriptive**: they explain which known risk segments a customer falls into. They show association, not cause.

In [ ]:
driver_features = [c for c in X.columns]
seg_labels = pd.DataFrame(index=X.index)
for c in driver_features:
    s = X[c]
    seg_labels[c] = pd.qcut(s, 4, duplicates="drop").astype(str) if (s.dtype.kind in "if" and s.nunique() > 8) else s.astype(str)

risky_segments = {}
for c in seg_labels:
    g = y.groupby(seg_labels[c]).agg(["mean", "size"])
    risky_segments[c] = set(g[(g["mean"] >= 1.25 * BASE_RATE) & (g["size"] >= 100)].index)

drivers = pd.Series("", index=X.index)
for c, segs in risky_segments.items():
    m = seg_labels[c].isin(segs)
    drivers[m] = drivers[m] + c + "=" + seg_labels.loc[m, c] + "; "
scored["RiskDrivers"] = drivers.str.rstrip("; ").replace("", "none")

print("Risky segments found:")
for c, segs in risky_segments.items():
    if segs: print(f"  {c}: {sorted(segs)}")

## Step 27. Export the high-risk customers to CSV
Saves two files:

| File | Content |
|---|---|
| `high_risk_customers.csv` | only the **High-risk** customers, with probability, expected balance at risk, risk drivers and key fields, sorted so the **biggest money at risk is first** |
| `all_customer_risk_scores.csv` | every customer with their probability and tier |

`ActualChurn` is included so you can check the model against what really happened. Surnames are left out. In Colab both files download to your computer.

In [ ]:
id_col = "CustomerId" if "CustomerId" in df.columns else ("CustomerID" if "CustomerID" in df.columns else None)
keep_cols = [c for c in [id_col, "Geography", "Gender", "Age", "Tenure", "CreditScore", "Balance", "EstimatedSalary",
                         "NumOfProducts", "HasCrCard", "IsActiveMember", "Satisfaction Score", "Card Type", "Point Earned"]
             if c and c in scored.columns]

out_cols = keep_cols + ["ChurnProbability", "RiskTier", "ExpectedBalanceAtRisk", "RiskDrivers"]
high_risk = (scored[scored.RiskTier == "High"][out_cols + ["Churn"]]
             .rename(columns={"Churn": "ActualChurn"})
             .sort_values("ExpectedBalanceAtRisk", ascending=False))
high_risk.insert(0, "PriorityRank", range(1, len(high_risk) + 1))

high_risk.to_csv("high_risk_customers.csv", index=False)
scored[out_cols + ["Churn"]].rename(columns={"Churn": "ActualChurn"}).to_csv("all_customer_risk_scores.csv", index=False)
joblib.dump(final_model, "churn_model.joblib")

print(f"Saved high_risk_customers.csv ({len(high_risk):,} customers), all_customer_risk_scores.csv and churn_model.joblib")
display(high_risk.head(15))
try:
    from google.colab import files
    for f in ["high_risk_customers.csv", "all_customer_risk_scores.csv"]:
        files.download(f)
except ImportError:
    pass

---
# How to read the results
1. **Step 18:** the model with the highest **Test AUC** wins. If several are within about 0.01, prefer the simpler one.
2. **Step 19:** use the **FP** and **TN** columns to see false alarms. A high FPR means many retention offers wasted on customers who would have stayed.
3. **Step 20:** choose a threshold that matches your budget for offers.
5. **Steps 24-26:** act on the **High** tier, starting with the largest `ExpectedBalanceAtRisk`, and use `RiskDrivers` to pick the right action for each customer.

**Limits.** A moderate AUC means the model ranks groups of customers usefully but cannot call any single customer with certainty. The data is a single snapshot, so results show association, not cause. Retrain regularly on fresh data.

# Glossary
| Term | Meaning |
|---|---|
| True negative (TN) | predicted to stay and stayed |
| False positive (FP) | predicted to leave but stayed: a wasted offer |
| False negative (FN) | predicted to stay but left: a missed churner |
| True positive (TP) | predicted to leave and left |
| False positive rate | FP ÷ (FP + TN): share of stayers wrongly flagged |
| Specificity | TN ÷ (TN + FP): share of stayers correctly left alone |
| Precision / Recall | of those flagged, the share who leave / of all who leave, the share caught |
| ROC-AUC | how well the model ranks churners above stayers: 0.5 = guessing, 1.0 = perfect |
| Leakage | using information the bank would not have before the customer decides |
| Lift | a group's churn rate divided by the overall rate |